# PINN Thermal Surrogate — Geometry 1

**What this trains:** A Physics-Informed Neural Network (FourierPINN) that predicts the 3D temperature field in a single-die 2D thermal stack (10mm × 10mm die, 6 layers including TIM2, total height 6400 µm).

**Architecture:** Fourier feature encoding + 6 residual MLP blocks, 807k parameters, SiLU activations, LayerNorm. Training uses curriculum staging: data-only → add PDE → NTK-adaptive weights.

## Kaggle Dataset Setup

Before running this notebook, add **two datasets** to the notebook:

1. **Source code** — upload the `src/` directory from the Thermo project as a Kaggle dataset.
   - Dataset slug suggestion: `thermo-pinn-src`
   - The dataset root should contain the `src/` folder (i.e., `/kaggle/input/thermo-pinn-src/src/` should exist)

2. **3D-ICE training data** — upload the `.npz` files from `data/3d-ice/` as a Kaggle dataset.
   - Dataset slug suggestion: `3dice-thermal-data`
   - Only geometry1 files are needed for this notebook (`geometry1_train_*.npz`, `geometry1_test_*.npz`)
   - Full dataset (all geometries) also works

**Expected training time on T4 GPU:** ~2–4 hours (8000 epochs, 15 training scenarios)  
**Expected val MAE:** < 2 K against 3D-ICE ground truth

In [ ]:
import subprocess, sys

# Install any missing packages (PyYAML is the only non-standard dep)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import os
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import sys
from pathlib import Path

# ── Kaggle dataset paths ────────────────────────────────────────────────────
# Adjust these slugs to match the datasets you attached to this notebook.
SRC_DATASET   = 'thermo-pinn-src'    # dataset containing the src/ folder
DATA_DATASET  = '3dice-thermal-data' # dataset containing the .npz files

SRC_ROOT  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_ROOT = Path(f'/kaggle/input/{DATA_DATASET}')
OUT_DIR   = Path('/kaggle/working/checkpoints/geometry1')

# Add source root to path so we can import src.*
sys.path.insert(0, str(SRC_ROOT))

# Verify
assert SRC_ROOT.exists(),  f'Source dataset not found at {SRC_ROOT}. Check SRC_DATASET slug.'
assert DATA_ROOT.exists(), f'Data dataset not found at {DATA_ROOT}. Check DATA_DATASET slug.'
print('Source root:', SRC_ROOT)
print('Data root:  ', DATA_ROOT)

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import logging
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import ThermalDataset, NormStats, compute_norm_stats
from src.pinn.model import build_model
from src.pinn.trainer import Trainer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
print('Imports OK')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
GEOM_NAME      = 'geometry1'
FOURIER_SIGMA  = 10.0    # Fourier feature bandwidth — 10 for geometry1 (no TSVs)
HIDDEN_DIM     = 256     # MLP width (256 = full, 128 = cpu-fast)
N_RES_BLOCKS   = 6       # residual blocks (6 = full, 4 = cpu-fast)
N_COL          = 20000   # collocation points per training step
EPOCHS         = 8000
LR             = 3e-4
LOG_INTERVAL   = 200     # print loss every N epochs
VAL_INTERVAL   = 1       # validate + save best checkpoint every N epochs
                         # 1 = every epoch — geometry1_best.pt stays current if session crashes

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
print(f'Config: hidden={HIDDEN_DIM} blocks={N_RES_BLOCKS} n_col={N_COL} epochs={EPOCHS}')

In [ ]:
# ── Geometry ─────────────────────────────────────────────────────────────────
geometry = get_geometry_by_name(GEOM_NAME)
geometries = {GEOM_NAME: geometry}

print(geometry.summary())

In [ ]:
# ── Data files ───────────────────────────────────────────────────────────────
def collect_files(data_dir: Path, geom: str, split: str):
    files = sorted(data_dir.rglob(f'{geom}_{split}_*.npz'))
    if not files:
        files = sorted(data_dir.glob(f'{geom}_{split}_*.npz'))
    return files

train_files = collect_files(DATA_ROOT, GEOM_NAME, 'train')
test_files  = collect_files(DATA_ROOT, GEOM_NAME, 'test')

assert train_files, f'No training files found in {DATA_ROOT}. Check DATA_DATASET slug and file naming.'
print(f'Training files : {len(train_files)}')
print(f'Test files     : {len(test_files)}')
print('First train file:', train_files[0].name)

In [ ]:
# ── Normalization statistics ──────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
norm_path = OUT_DIR / 'norm_stats.json'

if norm_path.exists():
    norm_stats = NormStats.load(norm_path)
    print('Loaded existing norm stats from', norm_path)
else:
    print('Computing norm stats over', len(train_files), 'training files...')
    norm_stats = compute_norm_stats(train_files, geometries)
    norm_stats.save(norm_path)
    print('Saved norm stats to', norm_path)

print(f'T range    : [{norm_stats.T_min:.1f}, {norm_stats.T_max:.1f}] K')
print(f'power_mean : {norm_stats.power_mean:.3e} W/m³')
print(f'power_std  : {norm_stats.power_std:.3e} W/m³')
print(f'geom extents: {norm_stats.geom_extents}')

In [ ]:
# ── Datasets ──────────────────────────────────────────────────────────────────
print('Loading training dataset...')
train_dataset = ThermalDataset(train_files, geometries, norm_stats)

val_files = test_files if test_files else train_files[-3:]
print('Loading validation dataset...')
val_dataset = ThermalDataset(val_files, geometries, norm_stats)

print(f'Train: {len(train_dataset)} scenarios')
print(f'Val  : {len(val_dataset)} scenarios')

# Show a sample scenario to confirm data loaded correctly
sc = train_dataset.scenarios[0]
print(f'\nSample scenario: {sc.name}')
print(f'  coords shape : {sc.coords.shape}  (N x 3 normalised)')
print(f'  temp range   : [{sc.temp.min():.3f}, {sc.temp.max():.3f}] normalised')
print(f'  htc          : {sc.htc:.0f} W/m²K  (norm={sc.htc_norm:.3f})')
print(f'  t_amb        : {sc.t_amb_K - 273.15:.1f} °C  (norm={sc.t_amb_norm:.3f})')
print(f'  tsv_frac     : {sc.tsv_frac}')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
n_layers = len(geometry.layers)
model = build_model(
    n_layers=n_layers,
    fourier_sigma=FOURIER_SIGMA,
    hidden_dim=HIDDEN_DIM,
    n_res_blocks=N_RES_BLOCKS,
    device=DEVICE,
)

n_params = sum(p.numel() for p in model.parameters())
print(f'FourierPINN: {n_params:,} parameters')
print(f'  n_layers={n_layers}, fourier_sigma={FOURIER_SIGMA}')
print(f'  hidden_dim={HIDDEN_DIM}, n_res_blocks={N_RES_BLOCKS}')
print(model)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    geometry=geometry,
    norm_stats=norm_stats,
    train_data=train_dataset,
    val_data=val_dataset,
    output_dir=OUT_DIR,
    n_col=N_COL,
    epochs=EPOCHS,
    lr=LR,
    device=DEVICE,
    log_interval=LOG_INTERVAL,
    val_interval=VAL_INTERVAL,
)

best_ckpt = trainer.train()
print('\nTraining complete. Best checkpoint:', best_ckpt)

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
history = trainer.history
epochs_logged = history['epoch']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'PINN Training — {GEOM_NAME}', fontsize=13)

# Loss components
ax = axes[0]
ax.semilogy(epochs_logged, history['train_data'], label='Data', color='steelblue')
ax.semilogy(epochs_logged, history['train_pde'],  label='PDE',  color='tomato')
ax.semilogy(epochs_logged, history['train_bc'],   label='BC',   color='forestgreen')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)')
ax.set_title('Training Loss Components'); ax.legend(); ax.grid(alpha=0.3)

# Total train loss
ax = axes[1]
ax.semilogy(epochs_logged, history['train_total'], color='darkorange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Total loss (log scale)')
ax.set_title('Total Training Loss'); ax.grid(alpha=0.3)

# Validation MAE
ax = axes[2]
ax.plot(epochs_logged, history['val_mae_K'], color='purple')
best_mae = min(history['val_mae_K'])
best_ep  = epochs_logged[history['val_mae_K'].index(best_mae)]
ax.axhline(best_mae, linestyle='--', color='purple', alpha=0.5,
           label=f'Best {best_mae:.2f} K @ epoch {best_ep}')
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE (K)')
ax.set_title('Validation MAE'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val MAE: {best_mae:.3f} K at epoch {best_ep}')

In [ ]:
# ── Evaluation on test set ────────────────────────────────────────────────────
# Load best checkpoint
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

T_range = norm_stats.T_max - norm_stats.T_min
results = []

with torch.no_grad():
    for sc in val_dataset.scenarios:
        htc_t  = torch.tensor(sc.htc_norm,   dtype=torch.float32, device=DEVICE)
        tamb_t = torch.tensor(sc.t_amb_norm, dtype=torch.float32, device=DEVICE)
        tsv_t  = torch.tensor(sc.tsv_frac,   dtype=torch.float32, device=DEVICE)

        T_pred_norm = model(sc.coords, sc.layer_ids, sc.power, htc_t, tamb_t, tsv_t)
        T_pred_K = T_pred_norm.cpu().numpy() * T_range + norm_stats.T_min
        T_true_K = sc.temp.cpu().numpy()     * T_range + norm_stats.T_min

        err = np.abs(T_pred_K - T_true_K)
        results.append({
            'name':    sc.name,
            'mae_K':   float(err.mean()),
            'p95_K':   float(np.percentile(err, 95)),
            'max_K':   float(err.max()),
            'T_max_pred': float(T_pred_K.max()),
            'T_max_true': float(T_true_K.max()),
        })

# Print table
print(f'{"Scenario":<35} {"MAE(K)":>8} {"P95(K)":>8} {"Max(K)":>8} {"T_max pred":>12} {"T_max true":>12}')
print('-' * 90)
for r in results:
    print(f'{r["name"]:<35} {r["mae_K"]:>8.2f} {r["p95_K"]:>8.2f} {r["max_K"]:>8.2f} '
          f'{r["T_max_pred"]:>12.1f} {r["T_max_true"]:>12.1f}')
print('-' * 90)
maes = [r['mae_K'] for r in results]
print(f'{"MEAN":<35} {np.mean(maes):>8.2f}')
print(f'{"MAX":<35} {np.max(maes):>8.2f}')

In [ ]:
# ── Predicted vs ground-truth temperature map (first test scenario) ───────────
sc = val_dataset.scenarios[0]
with torch.no_grad():
    htc_t  = torch.tensor(sc.htc_norm,   dtype=torch.float32, device=DEVICE)
    tamb_t = torch.tensor(sc.t_amb_norm, dtype=torch.float32, device=DEVICE)
    tsv_t  = torch.tensor(sc.tsv_frac,   dtype=torch.float32, device=DEVICE)
    T_pred_norm = model(sc.coords, sc.layer_ids, sc.power, htc_t, tamb_t, tsv_t)

T_pred_K = T_pred_norm.cpu().numpy() * T_range + norm_stats.T_min
T_true_K = sc.temp.cpu().numpy()     * T_range + norm_stats.T_min
err_K    = T_pred_K - T_true_K

coords = sc.coords.cpu().numpy()  # (N, 3) normalised

# Slice at mid-z of die layer (layer index for die in geometry1 = 4)
# Find the die layer
die_layer_idx = next(i for i, l in enumerate(geometry.layers) if l.is_active)
mask = sc.layer_ids.cpu().numpy() == die_layer_idx

if mask.sum() > 100:
    cx, cy = coords[mask, 0], coords[mask, 1]
    T_pred_slice = T_pred_K[mask]
    T_true_slice = T_true_K[mask]
    err_slice    = err_K[mask]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'Die-layer temperature — {sc.name}', fontsize=12)
    vmin, vmax = min(T_true_slice.min(), T_pred_slice.min()), max(T_true_slice.max(), T_pred_slice.max())

    sc0 = axes[0].scatter(cx, cy, c=T_true_slice, cmap='inferno', s=1, vmin=vmin, vmax=vmax)
    axes[0].set_title('Ground truth (3D-ICE)'); axes[0].set_aspect('equal')
    plt.colorbar(sc0, ax=axes[0], label='T (K)')

    sc1 = axes[1].scatter(cx, cy, c=T_pred_slice, cmap='inferno', s=1, vmin=vmin, vmax=vmax)
    axes[1].set_title('PINN prediction'); axes[1].set_aspect('equal')
    plt.colorbar(sc1, ax=axes[1], label='T (K)')

    err_abs = np.abs(err_slice)
    sc2 = axes[2].scatter(cx, cy, c=err_abs, cmap='RdYlGn_r', s=1, vmin=0, vmax=err_abs.max())
    axes[2].set_title(f'|Error|  MAE={err_abs.mean():.2f} K'); axes[2].set_aspect('equal')
    plt.colorbar(sc2, ax=axes[2], label='|T_pred − T_true| (K)')

    for ax in axes:
        ax.set_xlabel('x (norm)'); ax.set_ylabel('y (norm)')

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'die_temperature_map.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Not enough die-layer points in the first scenario to plot a map.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
import json

summary = {
    'geometry':      GEOM_NAME,
    'n_params':      n_params,
    'hidden_dim':    HIDDEN_DIM,
    'n_res_blocks':  N_RES_BLOCKS,
    'fourier_sigma': FOURIER_SIGMA,
    'epochs':        EPOCHS,
    'best_val_mae_K': best_mae,
    'best_epoch':    best_ep,
    'test_mae_mean_K': float(np.mean(maes)),
    'test_mae_max_K':  float(np.max(maes)),
    'checkpoint':    str(best_ckpt),
}

with open(OUT_DIR / 'training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Training summary:')
print(json.dumps(summary, indent=2))